# 2.1.1 SDDB 不同梦境中积极和负面内容分布

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

SDDB = pd.read_csv("SDDB_VADER.csv")
survey_order = sorted(SDDB["Survey Name"].unique())

fig, axes = plt.subplots(ncols=3, figsize=(18, 24), gridspec_kw={'width_ratios': [1, 0.52, 1], 'wspace': 0})
ax_neg, ax_label, ax_pos = axes

# neg
sns.stripplot(x="neg", 
              y="Survey Name", 
              data=SDDB, 
              ax=ax_neg, 
              order=survey_order, 
              color="royalblue", 
              alpha=0.5, 
              jitter=0.2, 
              size=4)

ax_neg.invert_xaxis()
ax_neg.set_ylabel("") 
ax_neg.set_xlabel("Negative Sentiment (Neg)", fontsize=14, fontweight='bold')
ax_neg.tick_params(axis='y', left=False, labelleft=False)
ax_neg.spines['right'].set_visible(False)
ax_neg.grid(axis='x', linestyle='--', alpha=0.6)

# pos
sns.stripplot(x="pos", 
              y="Survey Name", 
              data=SDDB, 
              ax=ax_pos, 
              order=survey_order, 
              color="crimson", 
              alpha=0.5, 
              jitter=0.2, 
              size=4)

ax_pos.set_ylabel("")
ax_pos.set_xlabel("Positive Sentiment (Pos)", fontsize=14, fontweight='bold')
ax_pos.tick_params(axis='y', left=False, labelleft=False)
ax_pos.spines['left'].set_visible(False)
ax_pos.grid(axis='x', linestyle='--', alpha=0.6)

# middle
ax_label.axis('off')
ax_label.set_ylim(ax_neg.get_ylim())
ax_label.set_xlim(0, 1)
for i, name in enumerate(survey_order):
    ax_label.text(0.5, i, name, ha='center', va='center', fontsize=11)

# plot
plt.suptitle("SDDB Negative vs. Positive Sentiment Distribution", fontsize=20, fontweight='bold', y=0.92)
plt.tight_layout(rect=[0, 0, 1, 0.9]) 
plt.show()

# 2.1.2 DB 不同梦境中积极和负面内容分布

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

DB = pd.read_csv("DB_VADER.csv")
survey_order = sorted(DB["Survey Name"].unique())

fig, axes = plt.subplots(ncols=3, figsize=(18, 24), gridspec_kw={'width_ratios': [1, 0.52, 1], 'wspace': 0})
ax_neg, ax_label, ax_pos = axes

# neg
sns.stripplot(x="neg", 
              y="Survey Name", 
              data=DB, 
              ax=ax_neg, 
              order=survey_order, 
              color="royalblue", 
              alpha=0.5, 
              jitter=0.2, 
              size=4)

ax_neg.invert_xaxis()
ax_neg.set_ylabel("") 
ax_neg.set_xlabel("Negative Sentiment (Neg)", fontsize=14, fontweight='bold')
ax_neg.tick_params(axis='y', left=False, labelleft=False)
ax_neg.spines['right'].set_visible(False)
ax_neg.grid(axis='x', linestyle='--', alpha=0.6)

# pos
sns.stripplot(x="pos", 
              y="Survey Name", 
              data=DB, 
              ax=ax_pos, 
              order=survey_order, 
              color="crimson", 
              alpha=0.5, 
              jitter=0.2, 
              size=4)

ax_pos.set_ylabel("")
ax_pos.set_xlabel("Positive Sentiment (Pos)", fontsize=14, fontweight='bold')
ax_pos.tick_params(axis='y', left=False, labelleft=False)
ax_pos.spines['left'].set_visible(False)
ax_pos.grid(axis='x', linestyle='--', alpha=0.6)

# middle
ax_label.axis('off')
ax_label.set_ylim(ax_neg.get_ylim())
ax_label.set_xlim(0, 1)
for i, name in enumerate(survey_order):
    ax_label.text(0.5, i, name, ha='center', va='center', fontsize=11)

# plot
plt.suptitle("DB Negative vs. Positive Sentiment Distribution", fontsize=20, fontweight='bold', y=0.92)
plt.tight_layout(rect=[0, 0, 1, 0.9]) 
plt.show()

# 2.2.1 性别对梦境内容的影响

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ast

df_vader = pd.read_csv('DB_VADER.csv')
df_spacy = pd.read_csv('DB_SPACY.csv')
df_empath = pd.read_csv('DB_EMPATH.csv')

if 'word_count' in df_spacy.columns and 'Word_Count' in df_spacy.columns:
    df_spacy = df_spacy.drop(columns=['word_count'])

print("2. 正在拼接并进行特征工程...")
merged_df = pd.concat([df_vader, df_spacy, df_empath], axis=1)
merged_df.columns = merged_df.columns.str.lower().str.replace(' ', '_')
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

def count_items(item):
    if pd.isna(item): return 0
    if isinstance(item, (int, float)): return item
    if isinstance(item, str):
        if item.startswith('['): 
            try: return len(ast.literal_eval(item))
            except: return 0
        else:
            return len([x for x in item.split(',') if x.strip()])
    return 0

merged_df['verb_count'] = merged_df.get('action_verbs', pd.Series([0]*len(merged_df))).apply(count_items)
merged_df['adj_count'] = merged_df.get('adjectives', pd.Series([0]*len(merged_df))).apply(count_items)
merged_df = merged_df[merged_df['word_count'] > 0].copy()

merged_df['verb_ratio'] = merged_df['verb_count'] / merged_df['word_count']
merged_df['adj_ratio'] = merged_df['adj_count'] / merged_df['word_count']

if 'difference' not in merged_df.columns and 'pos' in merged_df.columns:
    merged_df['difference'] = merged_df['pos'] - merged_df['neg']

features = [
    'verb_ratio', 'adj_ratio', 'violence', 'negative_emotion'
]
dataset_col = 'series'


def plot_experiment(df, group_column, title, palette="Set2"):
    plot_data = df.dropna(subset=[group_column]).copy()
    order = sorted(plot_data[group_column].unique())
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    fig.suptitle(title, fontsize=18, y=1.02, fontweight='bold')
    for i, feat in enumerate(features):
        ax = axes[i // 2, i % 2]
        if feat in plot_data.columns:
            sns.barplot(
                data=plot_data, x=group_column, y=feat, 
                order=order, ax=ax, palette=palette, capsize=.1, errorbar=None
            )
        ax.set_title(f'{feat.upper()}', fontsize=12, fontweight='bold')
        ax.set_ylabel(''); ax.set_xlabel('')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right', fontsize=10)
        
    plt.tight_layout()
    plt.show()

gender_map = {
    'norms-f': 'Norms (F)', 'norms-m': 'Norms (M)',
    'college-f': 'College 90s (F)', 'college-m': 'College 90s (M)',
    'zurich-f.de': 'Swiss Child (F)', 'zurich-m.de': 'Swiss Child (M)',
    'blind-f': 'Blind (F)', 'blind-m': 'Blind (M)'
}

merged_df['Exp_A_Gender'] = merged_df[dataset_col].map(gender_map)

print("3. 正在生成 2x2 精美图谱...")
plot_experiment(merged_df, 'Exp_A_Gender', 'Experiment A: Cross-Cultural Gender Differences', palette='coolwarm')

# 2.2.2 疫情对梦境内容的影响

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df_vader = pd.read_csv('DB_VADER.csv')
df_spacy = pd.read_csv('DB_SPACY.csv')
df_empath = pd.read_csv('DB_EMPATH.csv')

if 'word_count' in df_spacy.columns and 'Word_Count' in df_spacy.columns:
    df_spacy = df_spacy.drop(columns=['word_count'])

merged_df = pd.concat([df_vader, df_spacy, df_empath], axis=1)
merged_df.columns = merged_df.columns.str.lower().str.replace(' ', '_')
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]
merged_df = merged_df.rename(columns={'word_count': 'word_count'})

time_map = {
    '2010 Demographic Survey': '2010 Baseline',
    '2012 Demographic Survey': '2012 Baseline',
    '2013 Demographic Survey Summer': '2013 Baseline',
    '2013 Demographic Survey Winter': '2013 Baseline',
    '2020 Pandemic April Survey': '2020 Apr (Pandemic)',
    '2020 Pandemic May Survey': '2020 May (Pandemic)'
}

# 应用映射（假设你的数据框叫 analysis_df，且包含调查名称的列叫 'dataset'）
merged_df['Time_Group'] = merged_df['dataset'].map(time_map)

features = [
    'word_count', 'health', 'negative_emotion'
]

time_data = merged_df.dropna(subset=['Time_Group']).copy()
time_order = [
    '2010 Baseline', '2012 Baseline', '2013 Baseline', 
    '2020 Apr (Pandemic)', '2020 May (Pandemic)'
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle('Chronological Shift: 2010-2013 Baselines vs. 2020 Pandemic Crises', fontsize=20, y=1.02)

for i, feat in enumerate(features):
    ax = axes[i]
    sns.barplot(
        data=time_data, x='Time_Group', y=feat, 
        order=time_order, ax=ax, palette='viridis', capsize=.1
    )
    ax.set_title(f'{feat.upper()}', fontsize=14, fontweight='bold')
    
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel(feat)

plt.tight_layout()
plt.show()

# 2.2.3 社会问题对梦境内容的影响

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df_vader = pd.read_csv('SDDB_VADER.csv')
df_spacy = pd.read_csv('SDDB_SPACY.csv')
df_empath = pd.read_csv('SDDB_EMPATH.csv')

if 'word_count' in df_spacy.columns and 'Word_Count' in df_spacy.columns:
    df_spacy = df_spacy.drop(columns=['word_count'])

merged_df = pd.concat([df_vader, df_spacy, df_empath], axis=1)
merged_df.columns = merged_df.columns.str.lower().str.replace(' ', '_')
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]
merged_df = merged_df.rename(columns={'word_count': 'word_count'})

culture_map = {
    'Krippner International Collection, 1990-2005': 'International Baseline', 
    '2020 Racial Justice Survey': 'Racial Justice 2020'
}
merged_df['Culture_Group'] = merged_df['Survey Name'].map(culture_map)
culture_data = merged_df.dropna(subset=['Culture_Group']).copy()

features = [
    'word_count', 'difference', 
    'violence', 'family', 
    'negative_emotion', 'friends'
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Social Context: International Baseline vs. 2020 Racial Justice Movement', fontsize=20, y=1.02, fontweight='bold')

for i, feat in enumerate(features):
    ax = axes[i//3, i%3]
    sns.barplot(
        data=culture_data, x='Culture_Group', y=feat, 
        ax=ax, palette='magma', capsize=.1, errwidth=1.5
    )
    ax.set_title(f'{feat.upper()}', fontsize=14, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(feat) 

plt.tight_layout()
plt.show()